# Train YOLOv8 on playing cards

This notebook fine-tunes YOLOv8n on the Kaggle playing-cards object detection dataset (76 classes). After training, you upload `best.pt` to a new GitHub release `v0.2.0-cards` in the ramai repo, and update `DOWNLOAD_URL` in `rami/vision/pretrained.py`.

**Runtime:** ~30 min on free Colab GPU (T4).
**Output:** `best.pt` (~6 MB), `mAP50` reported on test set.

## Cell 1 — Install + imports

In [ ]:
!pip install -q ultralytics kaggle 2>&1 | tail -3

import os, json, shutil
from pathlib import Path
from ultralytics import YOLO
import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU! Go to Runtime → Change runtime type → T4 GPU.')

## Cell 2 — Download the Kaggle dataset

Upload your `kaggle.json` (from kaggle.com → Account → Create New API Token) when prompted.

In [ ]:
from google.colab import files
if not Path('/root/.kaggle/kaggle.json').exists():
    print('Upload kaggle.json...')
    files.upload()  # prompts for kaggle.json
    !mkdir -p /root/.kaggle && mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json

# Download the playing cards dataset
# The most popular one: 'hueyning/card-detection' or 'landryke/playing-cards-bentley'
# Adjust the dataset slug as needed.
!kaggle datasets download -d hueyning/card-detection -p /content/cards --unzip 2>&1 | tail -5

# Check structure
cards_dir = Path('/content/cards')
print(f'Cards dir: {cards_dir}')
if cards_dir.exists():
    for p in cards_dir.iterdir()[:5]:
        print(f'  {p.name}')

## Cell 3 — Prepare data.yaml

The dataset should have a `data.yaml` file defining the class names (76 classes for cards) and train/val paths. Adjust paths if necessary.

In [ ]:
data_yaml = cards_dir / 'data.yaml'
if not data_yaml.exists():
    # Search for it
    candidates = list(cards_dir.rglob('data.yaml')) + list(cards_dir.rglob('*.yaml'))
    if candidates:
        data_yaml = candidates[0]
    else:
        print('⚠ No data.yaml found. You may need to create one manually.')
        print('Expected format:')
        print('  path: /content/cards')
        print('  train: images/train')
        print('  val: images/val')
        print('  names:')
        print('    0: AS, 1: 2S, ... 75: Joker')

print(f'Using data.yaml: {data_yaml}')
if data_yaml.exists():
    print(data_yaml.read_text())

## Cell 4 — Fine-tune YOLOv8n (50 epochs, ~30 min)

In [ ]:
model = YOLO('yolov8n.pt')  # bootstrap from COCO weights
results = model.train(
    data=str(data_yaml),
    epochs=50,
    imgsz=640,
    batch=32,
    project='runs/train',
    name='cards_yolov8n',
    device=0 if torch.cuda.is_available() else 'cpu',
)
print('Training complete.')

## Cell 5 — Report mAP50 on validation set

In [ ]:
metrics = model.val()
print(f'mAP50:        {metrics.box.map50:.4f}')
print(f'mAP50-95:     {metrics.box.map:.4f}')
print(f'Precision:    {metrics.box.mp:.4f}')
print(f'Recall:       {metrics.box.mr:.4f}')
print()
print('These numbers should be reported in the main README.')
print('Target: mAP50 > 0.90')

## Cell 6 — Export best.pt and prepare for upload

In [ ]:
best_pt = Path('runs/train/cards_yolov8n/weights/best.pt')
if best_pt.exists():
    shutil.copy(best_pt, '/content/best.pt')
    size_mb = best_pt.stat().st_size / (1024 * 1024)
    print(f'✓ best.pt exported: {size_mb:.1f} MB')
    print(f'  Path: /content/best.pt')
    print()
    print('Next steps:')
    print('  1. Download best.pt to your computer')
    print('  2. Create a GitHub release v0.2.0-cards in VitalCheffe/ramai')
    print('  3. Upload best.pt as a release asset')
    print('  4. Update DOWNLOAD_URL in rami/vision/pretrained.py to:')
    print('     https://github.com/VitalCheffe/ramai/releases/download/v0.2.0-cards/best.pt')
    print('  5. Commit + push')
else:
    print(f'✗ best.pt not found at {best_pt}')